In [ ]:
# ----------------------------
# Global Imports
# ----------------------------
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
from tqdm import tqdm
from intervaltree import Interval, IntervalTree
import random
import ast
from pathlib import Path
import statistics
from math import ceil
import torch
import h5py
import numpy as np
from collections import defaultdict
import os

In [ ]:
# ----------------------------
# Global Parameters
# ----------------------------
GFF_CLEAN_KEYWORD = "genbank"      # Keyword for filtering the GFF annotation lines
CHROMOSOME_PREFIX = "NC"           # Identifier prefix to filter chromosome records
PLANT_NAME = "Arabidopsis_thaliana"
random.seed(42)
working_dir = Path(
    f'/data/Step2_generate_training_data/{PLANT_NAME}')

In [ ]:
# ----------------------------
# Step 1: Filter STAR SJ.out.tab file for majority chromosomes (e.g., NC)
# ----------------------------
with open(f"{working_dir}/{PLANT_NAME}.SJ.out.tab", 'r') as input_file:
    with open(f"{working_dir}/{PLANT_NAME}.SJ.out.clean.tab", 'w') as output_file:
        for line in input_file:
            if line.startswith(CHROMOSOME_PREFIX):
                output_file.write(line)

In [ ]:
# ----------------------------
# Step 2: Process STAR junction list (extract intron start and end sites)
# ----------------------------
star_intron_start_set = set()
star_intron_end_set = set()

with open(f"{working_dir}/{PLANT_NAME}.SJ.out.clean.tab", 'r') as input_file:
    for line in input_file:
        fields = line.strip().split("\t")
        chromosome = fields[0]
        intron_start = int(fields[1])
        intron_end = int(fields[2])
        strand_flag = fields[3]
        intron_type = fields[4]  # '1' for noncanonical, '0' for canonical
        unique_mapped_read_count = int(fields[6])
        multiple_mapped_read_count = int(fields[7])
        overhang_count = int(fields[8])
        intron_length = intron_end - intron_start + 1

        # Filtering criteria based on provided thresholds
        if intron_type == '1' and unique_mapped_read_count < 3:
            continue
        if intron_length > 50000 and unique_mapped_read_count < 2:
            continue
        if intron_length > 100000 and unique_mapped_read_count < 3:
            continue
        if intron_length > 200000 and unique_mapped_read_count < 4:
            continue
        if intron_type == '1' and overhang_count < 30:
            continue
        if intron_type == '0' and overhang_count < 12:
            continue
        if unique_mapped_read_count < 10:
            continue

        # Format junction record based on strand information
        if strand_flag == "1":  # Positive strand
            junction_start_record = f'{chromosome}_+_{intron_start}'
            junction_end_record = f'{chromosome}_+_{intron_end}'
        elif strand_flag == "2":  # Negative strand
            junction_start_record = f'{chromosome}_-_{intron_end}'
            junction_end_record = f'{chromosome}_-_{intron_start}'
        else:
            continue

        star_intron_start_set.add(junction_start_record)
        star_intron_end_set.add(junction_end_record)

# Write STAR intron start and end junction sets to files
with open(f"{working_dir}/{PLANT_NAME}_junction_STAR_list_intron_start.txt", 'w') as out_file:
    for item in star_intron_start_set:
        out_file.write(f'{item}\n')

with open(f"{working_dir}/{PLANT_NAME}_junction_STAR_list_intron_end.txt", 'w') as out_file:
    for item in star_intron_end_set:
        out_file.write(f'{item}\n')

In [ ]:
# ----------------------------
# Step 3: Generate Clean Exon File and Create Gene-to-Transcript Dictionary
# ----------------------------
gene_transcript_dict = {}

# Filter GFF file by chromosome and keyword
with open(f'{working_dir}/{PLANT_NAME}.gff', 'r') as infile:
    with open(f'{working_dir}/{PLANT_NAME}_clean.gff', 'w') as outfile:
        for line in infile:
            if line.startswith(CHROMOSOME_PREFIX):
                parts = line.strip().split("\t")
                chromosome = parts[0]
                element_type = parts[2]
                start = int(parts[3])
                end = int(parts[4])
                strand = parts[6]
                annotation = parts[8].lower()
                record = f'{chromosome}\t{element_type}\t{start}\t{end}\t{strand}\t{annotation}\n'
                if GFF_CLEAN_KEYWORD in annotation:
                    outfile.write(record)

# Generate exon file from clean GFF
with open(f'{working_dir}/{PLANT_NAME}_clean.gff', 'r') as infile:
    with open(f'{working_dir}/{PLANT_NAME}_exon.gff', 'w') as outfile:
        for line in infile:
            if line.startswith(CHROMOSOME_PREFIX):
                parts = line.strip().split("\t")
                chromosome = parts[0]
                element_type = parts[1]
                start = int(parts[2])
                end = int(parts[3])
                strand = parts[4]
                annotation = parts[5]
                record = f'{chromosome}\t{element_type}\t{start}\t{end}\t{strand}\t{annotation}\n'
                if element_type == "exon":
                    outfile.write(record)

# Build gene-to-transcript dictionary from the exon file
with open(f'{working_dir}/{PLANT_NAME}_exon.gff', 'r') as infile:
    for line in infile:
        parts = line.strip().split("\t")
        annotation = parts[-1]
        transcript_id = annotation.split("parent=rna-")[1].split(";")[0]
        gene_id = annotation.split("geneid:")[1].split(";")[0].split(",")[0]
        if gene_id in gene_transcript_dict:
            if transcript_id not in gene_transcript_dict[gene_id]:
                gene_transcript_dict[gene_id].append(transcript_id)
        else:
            gene_transcript_dict[gene_id] = [transcript_id]

with open(f'{working_dir}/{PLANT_NAME}_gene_transcript.txt', 'w') as outfile:
    for gene_id, transcript_list in gene_transcript_dict.items():
        transcript_str = ','.join(transcript_list)
        outfile.write(f'{gene_id}\t{transcript_str}\n')

In [ ]:
# ----------------------------
# Step 4: Create Intron Junction Dictionaries
# ----------------------------
transcript_exon_dict = {}
gene_exon_dict = {}           # (Not further used in subsequent steps)
transcript_to_gene = {}       # Maps transcript ID to gene ID

# Map each transcript to its gene from the exon file
with open(f'{working_dir}/{PLANT_NAME}_exon.gff', 'r') as infile:
    for line in infile:
        parts = line.strip().split("\t")
        annotation = parts[-1]
        transcript_id = annotation.split("parent=rna-")[1].split(";")[0]
        gene_id = annotation.split("geneid:")[1].split(";")[0].split(",")[0]
        if transcript_id not in transcript_to_gene:
            transcript_to_gene[transcript_id] = gene_id

# Build transcript-to-exon dictionary (each transcript holds a list of exon (start, end) tuples)
with open(f'{working_dir}/{PLANT_NAME}_exon.gff', 'r') as infile:
    for line in infile:
        parts = line.strip().split("\t")
        chromosome = parts[0]
        start = int(parts[2])
        end = int(parts[3])
        strand = parts[4]
        annotation = parts[-1]
        transcript_id = annotation.split("parent=rna-")[1].split(";")[0]

        if transcript_id in transcript_exon_dict:
            transcript_exon_dict[transcript_id][2].append((start, end))
        else:
            transcript_exon_dict[transcript_id] = [
                chromosome, strand, [(start, end)]]

# Write the transcript exon dictionary for reference
with open(f"{working_dir}/{PLANT_NAME}_transcript_exon_dictionary.txt", 'w') as outfile:
    for transcript, data in transcript_exon_dict.items():
        outfile.write(f'{transcript}\t{data}\n')

# Generate intron regions for each transcript based on exon coordinates
transcript_intron_dict = {}
for transcript_id, data in transcript_exon_dict.items():
    chromosome, strand, exon_list = data
    sorted_exons = sorted(exon_list, key=lambda x: x[0])
    introns = []
    for i in range(len(sorted_exons) - 1):
        current_exon_end = sorted_exons[i][1]
        next_exon_start = sorted_exons[i + 1][0]
        intron_start = current_exon_end + 1
        intron_end = next_exon_start - 1
        introns.append((intron_start, intron_end))
    if strand == '-':
        introns = introns[::-1]
    transcript_intron_dict[transcript_id] = [chromosome, strand, introns]

# Build transcript intron junction dictionary (junction format: chromosome_strand_site)
transcript_intron_junction_dict = {}
for transcript_id, data in transcript_intron_dict.items():
    chromosome, strand, intron_list = data
    junction_set = transcript_intron_junction_dict.setdefault(
        transcript_id, set())
    for intron in intron_list:
        for junction_site in intron:
            junction_detail = f'{chromosome}_{strand}_{junction_site}'
            junction_set.add(junction_detail)

# Remove entries with empty junction sets and sort the junction lists
for tid in list(transcript_intron_junction_dict.keys()):
    if not transcript_intron_junction_dict[tid]:
        del transcript_intron_junction_dict[tid]
    else:
        transcript_intron_junction_dict[tid] = sorted(
            list(transcript_intron_junction_dict[tid]))

with open(f"{working_dir}/{PLANT_NAME}_transcript_intron_junction_dictionary.txt", 'w') as outfile:
    for tid, junctions in transcript_intron_junction_dict.items():
        if junctions:
            outfile.write(f'{tid}\t{junctions}\n')

# Build gene intron junction dictionary using transcript junctions and transcript-to-gene mapping
gene_intron_junction_dict = {}
for transcript_id, data in transcript_intron_dict.items():
    chromosome, strand, intron_list = data
    gene_id = transcript_to_gene[transcript_id]
    junction_set = gene_intron_junction_dict.setdefault(gene_id, set())
    for intron in intron_list:
        for junction_site in intron:
            junction_detail = f'{chromosome}_{strand}_{junction_site}'
            junction_set.add(junction_detail)

# Clean up and sort gene intron junctions
for gene_id in list(gene_intron_junction_dict.keys()):
    if not gene_intron_junction_dict[gene_id]:
        del gene_intron_junction_dict[gene_id]
    else:
        gene_intron_junction_dict[gene_id] = sorted(
            list(gene_intron_junction_dict[gene_id]))

with open(f"{working_dir}/{PLANT_NAME}_gene_intron_junction_dictionary.txt", 'w') as outfile:
    for gene_id, junctions in gene_intron_junction_dict.items():
        if junctions:
            outfile.write(f'{gene_id}\t{junctions}\n')

# Load STAR junction lists (both start and end)
star_junction_set = set()
with open(f"{working_dir}/{PLANT_NAME}_junction_STAR_list_intron_start.txt", 'r') as infile:
    star_junction_set.update(line.strip() for line in infile)
with open(f"{working_dir}/{PLANT_NAME}_junction_STAR_list_intron_end.txt", 'r') as infile:
    star_junction_set.update(line.strip() for line in infile)

# Process qualified genes: genes for which every intron junction is validated by STAR
with open(f'{working_dir}/{PLANT_NAME}_qualified_gene.txt', 'w') as outfile:
    for gene_id, junctions in tqdm(gene_intron_junction_dict.items(), desc="Processing genes"):
        if all(junction in star_junction_set for junction in junctions):
            outfile.write(f'{gene_id}\n')

# Visual annotation: mark each junction that is STAR validated with an asterisk (*)
with open(f"{working_dir}/{PLANT_NAME}_qualified_gene_visualization.txt", 'w') as outfile:
    for gene_id, junctions in tqdm(gene_intron_junction_dict.items(), desc="Processing genes"):
        annotated_junctions = [
            f'{junction}*' if junction in star_junction_set else junction for junction in junctions]
        outfile.write(f'{gene_id}\t{annotated_junctions}\n')

In [ ]:
# ----------------------------
# Step 5: Filter out AS Event I (A5SS/A3SS) for Annotation‐Qualified No‐AS Genes
# ----------------------------
# Define dictionaries to store gene boundaries, regions, strand/chromosome information, and transcript mappings.
# Temporary storage of gene boundary positions (as a set)
gene_range = dict()
# Final gene region: [chromosome, [min, max]]
gene_region = dict()
# Mapping of gene_id to its region (same as gene_region)
intron_gene_region = dict()
gene_strand = dict()               # Mapping of gene_id to strand
gene_chromosome = dict()           # Mapping of gene_id to chromosome
gene_transcript = dict()           # Mapping of gene_id to list of transcript IDs

# Dictionaries to store annotated intron junctions from external files
plant_gene_intron_juction_dictionary = dict()
plant_transcript_intron_juction_dictionary = dict()

# Dictionary to hold final processed gene data (gene strand, region, intron junctions, etc.)
dictionary_for_gene_strand_range_intron_actualjunction = dict()

# Dictionaries for STAR validated intron junctions and junction read counts
STAR_validated_intron_start_dict, STAR_validated_intron_end_dict = {}, {}
junction_reads_count_dictionary = dict()

# ---------------------------------------------------------------------------
# Build junction_reads_count_dictionary from STAR SJ.out.clean.tab
# ---------------------------------------------------------------------------
with open(f'{working_dir}/{PLANT_NAME}.SJ.out.clean.tab', 'r') as filein:
    for line in filein:
        feature = line.strip().split("\t")
        chromosome = feature[0]
        intron_start = feature[1]
        intron_end = feature[2]
        strand = feature[3]
        count = int(feature[-3])
        # Normalize strand flag to symbol
        if strand == "1":
            strand = "+"
        elif strand == "2":
            strand = "-"
        else:
            strand = "0"
        intron_start_record = f'{chromosome}_{strand}_{intron_start}'
        intron_end_record = f'{chromosome}_{strand}_{intron_end}'
        junction_reads_count_dictionary[intron_start_record] = junction_reads_count_dictionary.get(
            intron_start_record, 0) + count
        junction_reads_count_dictionary[intron_end_record] = junction_reads_count_dictionary.get(
            intron_end_record, 0) + count

# ---------------------------------------------------------------------------
# Build gene_transcript mapping from gene_transcript.txt
# ---------------------------------------------------------------------------
with open(f'{working_dir}/{PLANT_NAME}_gene_transcript.txt', 'r') as filein:
    for line in filein:
        feature = line.strip().split("\t")
        geneid = feature[0]
        transcript_str = feature[-1]
        transcript_list = transcript_str.split(",")
        for item in transcript_list:
            gene_transcript.setdefault(geneid, []).append(item)

# ---------------------------------------------------------------------------
# Parse clean.gff to build gene boundaries (gene_range) using start and end positions
# ---------------------------------------------------------------------------
with open(f"{working_dir}/{PLANT_NAME}_clean.gff", 'r') as filein:
    for line in filein:
        feature = line.strip().split("\t")
        start = int(feature[2])
        end = int(feature[3])
        gene_id = feature[-1].split("geneid:")[1].split(",")[0].split(";")[0]
        gene_range.setdefault(gene_id, set()).update({start, end})

# Parse clean.gff to record chromosome and strand info for each gene
with open(f"{working_dir}/{PLANT_NAME}_clean.gff", 'r') as filein:
    for line in filein:
        feature = line.strip().split("\t")
        chromosome = feature[0]
        strand = feature[4]
        gene_id = feature[-1].split("geneid:")[1].split(",")[0].split(";")[0]
        gene_chromosome[gene_id] = chromosome
        gene_strand[gene_id] = strand

# Convert gene boundaries (set) to a sorted list and build gene_region dictionary
for gene_id in gene_range.keys():
    gene_range[gene_id] = sorted(list(gene_range[gene_id]))
for gene_id, region in gene_range.items():
    min_range = min(region)
    max_range = max(region)
    chromosome = gene_chromosome[gene_id]
    gene_region[gene_id] = [chromosome, [min_range, max_range]]

# Write gene_region to file (for reference)
with open("gene_region.txt", 'w') as outfile:
    for gene_id, data in gene_region.items():
        outfile.write(f'{gene_id}\t{data}\n')

# ---------------------------------------------------------------------------
# Build plant_gene_intron_juction_dictionary from gene_intron_junction_dictionary.txt
# ---------------------------------------------------------------------------
with open(f"{working_dir}/{PLANT_NAME}_gene_intron_junction_dictionary.txt", 'r') as filein:
    for line in filein:
        feature = line.strip().split("\t")
        gene_id = feature[0]
        generegion = gene_region[gene_id]
        intron_gene_region[gene_id] = generegion
        record = []
        data = ast.literal_eval(feature[-1])
        # Each item in data is a junction string (e.g., "chromosome_strand_position")
        for item in data:
            intron_site = int(item.split("_")[-1])
            record.append(intron_site)
        plant_gene_intron_juction_dictionary[gene_id] = record

# Build plant_transcript_intron_juction_dictionary from transcript_intron_junction_dictionary.txt
with open(f"{working_dir}/{PLANT_NAME}_transcript_intron_junction_dictionary.txt", 'r') as filein:
    for line in filein:
        feature = line.strip().split("\t")
        transcript_id = feature[0]
        record = []
        data = ast.literal_eval(feature[-1])
        for item in data:
            intron_site = int(item.split("_")[-1])
            record.append(intron_site)
        plant_transcript_intron_juction_dictionary[transcript_id] = record

# ---------------------------------------------------------------------------
# Load STAR validated intron sites into dictionaries by chromosome
# ---------------------------------------------------------------------------
with open(f"{working_dir}/{PLANT_NAME}_junction_STAR_list_intron_start.txt", 'r') as filein:
    for line in filein:
        parts = line.strip().split('_')
        chromosome = '_'.join(parts[:-2])
        position = int(parts[-1])
        strand = parts[2]
        STAR_validated_intron_start_dict.setdefault(
            chromosome, []).append(f'{position}_{strand}')

with open(f"{working_dir}/{PLANT_NAME}_junction_STAR_list_intron_end.txt", 'r') as filein:
    for line in filein:
        parts = line.strip().split('_')
        strand = parts[2]
        chromosome = '_'.join(parts[:-2])
        position = int(parts[-1])
        STAR_validated_intron_end_dict.setdefault(
            chromosome, []).append(f'{position}_{strand}')

# ---------------------------------------------------------------------------
# Process each gene: annotate STAR-validated intron junctions and add transcript info
# ---------------------------------------------------------------------------
for gene_id, data in tqdm(intron_gene_region.items(), desc="Loading"):
    chromosome_gene, gene_range_vals = data[0], data[1]
    geneid_strand = gene_strand[gene_id]
    start, end = gene_range_vals
    intron_start_set = set()
    intron_end_set = set()

    # Collect STAR-validated intron start sites within the gene region
    if chromosome_gene in STAR_validated_intron_start_dict:
        for site in STAR_validated_intron_start_dict[chromosome_gene]:
            position = int(site.split("_")[0])
            position_strand = site.split("_")[1]
            if position_strand == geneid_strand and start <= position <= end:
                intron_start_set.add(position)

    # Collect STAR-validated intron end sites within the gene region
    if chromosome_gene in STAR_validated_intron_end_dict:
        for site in STAR_validated_intron_end_dict[chromosome_gene]:
            position = int(site.split("_")[0])
            position_strand = site.split("_")[1]
            if position_strand == geneid_strand and start <= position <= end:
                intron_end_set.add(position)

    sorted_intron_start_list = sorted(intron_start_set)
    sorted_intron_end_list = sorted(intron_end_set)

    # Retrieve annotated introns using the original dictionary approach;
    # if the gene_id is missing, default to an empty list.
    try:
        annotated_intron = sorted(
            plant_gene_intron_juction_dictionary[gene_id])
    except KeyError:
        annotated_intron = []

    # Retrieve transcript list from gene_transcript
    transcript_list = gene_transcript[gene_id]
    overall_STAR_junction = sorted(
        sorted_intron_start_list + sorted_intron_end_list)

    # Build the record for this gene
    record = [chromosome_gene, geneid_strand, gene_range_vals, annotated_intron,
              sorted_intron_start_list, sorted_intron_end_list, overall_STAR_junction]

    overall_STAR_junction_with_count = []
    for junction in overall_STAR_junction:
        record_for_search = f"{chromosome_gene}_{geneid_strand}_{junction}"
        count = junction_reads_count_dictionary.get(record_for_search, '')
        overall_STAR_junction_with_count.append(f'{junction}:{count}')

    # Filter junctions based on read count threshold
    if not overall_STAR_junction_with_count:
        filtered_overall_STAR_junction_sites = []
    else:
        sites_counts = [item.split(":")
                        for item in overall_STAR_junction_with_count]
        counts = [int(count) for _, count in sites_counts]
        highest_count = max(int(count) for _, count in sites_counts)
        median_count = statistics.median(counts)
        threshold = 0.05 * median_count
        filtered_overall_STAR_junction_sites = sorted([int(site)
                                                       for site, count in sites_counts if int(count) >= threshold])
    record.append(filtered_overall_STAR_junction_sites)

    # Append transcript-specific intron junction information
    for transcript in transcript_list:
        intron_transcript = plant_transcript_intron_juction_dictionary.get(
            transcript, '')
        add_info = f'{transcript}:{intron_transcript}'
        record.append(add_info)

    dictionary_for_gene_strand_range_intron_actualjunction[gene_id] = record

# Write the final dictionary to file
with open(f'{working_dir}/{PLANT_NAME}_dictionary_for_gene_strand_range_intron_actualjunction.txt', 'w') as outfile:
    for geneid, info in dictionary_for_gene_strand_range_intron_actualjunction.items():
        outfile.write(f'{geneid}\t{info}\n')

In [ ]:
# ----------------------------
# Step 6: Filter AS Event II (IR & ES) for specific RNA types
# ----------------------------
chromosome_trees = {}
gene_region = {}
mRNA = set()
with open(f'{working_dir}/{PLANT_NAME}_clean.gff', 'r') as filein:
    for line in filein:
        feature = line.strip().split("\t")
        element_type = feature[1]
        info = feature[-1]
        if element_type == 'mRNA':
            geneid = info.split("geneid:")[1].split(",")[0]
            mRNA.add(geneid)

with open(f'{working_dir}/{PLANT_NAME}_mRNA_geneid.txt', 'w') as filein:
    for item in mRNA:
        filein.write(f'{item}\n')

with open(f"{working_dir}/gene_region.txt", 'r') as filein:
    for line in filein:
        feature = line.strip().split("\t")
        gene_id = feature[0]
        info = ast.literal_eval(feature[1])
        gene_region[gene_id] = info

for gene_id, (chromosome, [start, end]) in gene_region.items():
    if chromosome not in chromosome_trees:
        chromosome_trees[chromosome] = IntervalTree()
    chromosome_trees[chromosome].add(Interval(start, end, gene_id))


def find_gene_id_for_site(record, chromosome_trees):
    feature = record.split("_")
    chromosome = f'{feature[0]}_{feature[1]}'
    site_str = feature[2]
    site = int(site_str)
    if chromosome in chromosome_trees:
        overlapping_intervals = chromosome_trees[chromosome][site]
        for interval in overlapping_intervals:
            return interval.data
    return None


with open(f"{working_dir}/{PLANT_NAME}_IR_PSI.txt", 'r') as filein:
    with open(f'{working_dir}/{PLANT_NAME}_IR_detect_gene95%.txt', 'w') as fileout:
        for line in filein:
            feature = line.strip().split("\t")
            chromosome = feature[1]
            intron_position = feature[2]
            intron_start = int(intron_position.split("_")[0])
            clean_or_not = feature[6]
            E1_I = int(feature[7])
            I_E2 = int(feature[8])
            warning1 = feature[10]
            warning2 = feature[11]
            if feature[-1] != "-":
                PSI = float(feature[-1])
            else:
                continue
            if clean_or_not == "Clean" and E1_I >= 5 and I_E2 >= 5 and warning1 != "Low_Count" and warning2 != "Imbalance" and PSI >= 0.95:
                record = f'{chromosome}_{intron_start}'
                geneid = find_gene_id_for_site(record, chromosome_trees)
                if geneid != None:
                    fileout.write(f'{geneid}\n')

with open(f"{working_dir}/{PLANT_NAME}_ES_PSI.txt", 'r') as filein:
    with open(f'{working_dir}/{PLANT_NAME}_ES_detect_gene95%.txt', 'w') as fileout:
        for line in filein:
            feature = line.strip().split("\t")
            chromosome = feature[1]
            exon_position = feature[2]
            exon_start = int(intron_position.split("_")[0])
            clean_or_not = feature[6]
            E1_E2 = int(feature[9])
            warning1 = feature[10]
            warning2 = feature[11]
            if feature[-1] != "-":
                PSI = float(feature[-1])
            else:
                continue
            if clean_or_not == "Clean" and E1_E2 >= 5 and warning1 != "Low_Count" and warning2 != "Imbalance" and PSI <= 0.05:
                record = f'{chromosome}_{exon_start}'
                geneid = find_gene_id_for_site(record, chromosome_trees)
                if geneid != None:
                    fileout.write(f'{geneid}\n')

In [ ]:
# ----------------------------
# Step 7: Final Venn Set for Training Genes
# ----------------------------
STAR_qualified_genes = set()
AS53SS_gene_set = set()
no_AS53SS_gene_set = set()
IR_gene_set = set()
ES_gene_set = set()
mrna_gene_set = set()

with open(f'{working_dir}/{PLANT_NAME}_mRNA_geneid.txt', 'r') as infile:
    for line in infile:
        mrna_gene_set.add(line.strip())

with open(f"{working_dir}/{PLANT_NAME}_qualified_gene.txt", 'r') as infile:
    for line in infile:
        STAR_qualified_genes.add(line.strip())

with open(f'{working_dir}/{PLANT_NAME}_IR_detect_gene95%.txt', 'r') as infile:
    for line in infile:
        IR_gene_set.add(line.strip())

with open(f'{working_dir}/{PLANT_NAME}_ES_detect_gene95%.txt', 'r') as infile:
    for line in infile:
        ES_gene_set.add(line.strip())

with open(f"{working_dir}/{PLANT_NAME}_dictionary_for_gene_strand_range_intron_actualjunction.txt", 'r') as infile:
    for line in infile:
        parts = line.strip().split("\t")
        gene_id = parts[0]
        info = ast.literal_eval(parts[-1])
        intron_start_list = info[4]
        intron_end_list = info[5]
        annotated_junction = info[3]
        actual_junction = info[6]
        filtered_actual_junction = info[7]
        transcript_intron_info = info[8:]
        all_intron_lists = []
        if len(transcript_intron_info) != 1:
            for item in transcript_intron_info:
                try:
                    intron_list = [int(x.strip()) for x in item.split(
                        "[")[1].split("]")[0].split(",")]
                    all_intron_lists.append(intron_list)
                except:
                    intron_list = []
            # Check if all transcript intron lists are identical
            if all_intron_lists and all(set(intron_list) == set(all_intron_lists[0]) for intron_list in all_intron_lists):
                if len(intron_start_list) == len(intron_end_list):
                    if annotated_junction == filtered_actual_junction:
                        no_AS53SS_gene_set.add(gene_id)
                    else:
                        AS53SS_gene_set.add(gene_id)
                else:
                    AS53SS_gene_set.add(gene_id)
            else:
                AS53SS_gene_set.add(gene_id)
        else:
            if len(intron_start_list) == len(intron_end_list):
                if annotated_junction == filtered_actual_junction:
                    no_AS53SS_gene_set.add(gene_id)
                else:
                    AS53SS_gene_set.add(gene_id)
            else:
                AS53SS_gene_set.add(gene_id)

# Final gene lists after set intersections and differences
no_AS_genes_final_list = sorted(
    STAR_qualified_genes & no_AS53SS_gene_set & mrna_gene_set - IR_gene_set - ES_gene_set)
AS_genes_final_list = sorted(
    STAR_qualified_genes & AS53SS_gene_set & mrna_gene_set - IR_gene_set - ES_gene_set)

with open(f"{working_dir}/{PLANT_NAME}_no_AS_genes_final_list.txt", 'w') as outfile:
    for gene in no_AS_genes_final_list:
        outfile.write(f'{gene}\n')

with open(f"{working_dir}/{PLANT_NAME}_AS_genes_final_list.txt", 'w') as outfile:
    for gene in AS_genes_final_list:
        outfile.write(f'{gene}\n')

In [ ]:
# ----------------------------
# Step 8: Split Genes into Training and Testing Sets (80/20 split)
# ----------------------------
no_AS_genes_final_list_loaded = []
AS_genes_final_list_loaded = []

with open(f"{working_dir}/{PLANT_NAME}_no_AS_genes_final_list.txt", 'r') as infile:
    for line in infile:
        no_AS_genes_final_list_loaded.append(line.strip())

with open(f"{working_dir}/{PLANT_NAME}_AS_genes_final_list.txt", 'r') as infile:
    for line in infile:
        AS_genes_final_list_loaded.append(line.strip())

# Shuffle lists with fixed seed and calculate split index
random.shuffle(no_AS_genes_final_list_loaded)
random.shuffle(AS_genes_final_list_loaded)
no_AS_split_idx = int(0.8 * len(no_AS_genes_final_list_loaded))
AS_split_idx = int(0.8 * len(AS_genes_final_list_loaded))

no_AS_genes_train = no_AS_genes_final_list_loaded[:no_AS_split_idx]
no_AS_genes_test = no_AS_genes_final_list_loaded[no_AS_split_idx:]
AS_genes_train = AS_genes_final_list_loaded[:AS_split_idx]
AS_genes_test = AS_genes_final_list_loaded[AS_split_idx:]

with open(f"{working_dir}/{PLANT_NAME}_no_AS_genes_train.txt", 'w') as outfile:
    for gene in no_AS_genes_train:
        outfile.write(f'{gene}\n')

with open(f"{working_dir}/{PLANT_NAME}_no_AS_genes_test.txt", 'w') as outfile:
    for gene in no_AS_genes_test:
        outfile.write(f'{gene}\n')

with open(f"{working_dir}/{PLANT_NAME}_AS_gene_train.txt", 'w') as outfile:
    for gene in AS_genes_train:
        outfile.write(f'{gene}\n')

with open(f"{working_dir}/{PLANT_NAME}_AS_gene_test.txt", 'w') as outfile:
    for gene in AS_genes_test:
        outfile.write(f'{gene}\n')

In [ ]:
# ----------------------------
# Step 9: Define Intron Junction Positions within Genes and Partition by Set
# ----------------------------
no_AS_genes_train_list = []
no_AS_genes_test_list = []
AS_genes_train_list = []
AS_genes_test_list = []
AS_train_actual_junction = {}
AS_test_actual_junction = {}
no_AS_train_actual_junction = {}
no_AS_test_actual_junction = {}
other_gene_actual_junction = {}
mrna_gene_set_final = set()

with open(f'{working_dir}/{PLANT_NAME}_mRNA_geneid.txt', 'r') as infile:
    for line in infile:
        mrna_gene_set_final.add(line.strip())

with open(f"{working_dir}/{PLANT_NAME}_no_AS_genes_train.txt", 'r') as infile:
    for line in infile:
        no_AS_genes_train_list.append(line.strip())

with open(f"{working_dir}/{PLANT_NAME}_no_AS_genes_test.txt", 'r') as infile:
    for line in infile:
        no_AS_genes_test_list.append(line.strip())

with open(f"{working_dir}/{PLANT_NAME}_AS_gene_train.txt", 'r') as infile:
    for line in infile:
        AS_genes_train_list.append(line.strip())

with open(f"{working_dir}/{PLANT_NAME}_AS_gene_test.txt", 'r') as infile:
    for line in infile:
        AS_genes_test_list.append(line.strip())

with open(f'{working_dir}/{PLANT_NAME}_dictionary_for_gene_strand_range_intron_actualjunction.txt', 'r') as infile:
    for line in infile:
        parts = line.strip().split("\t")
        gene_id = parts[0]
        info = ast.literal_eval(parts[1])
        if gene_id in AS_genes_train_list:
            AS_train_actual_junction[gene_id] = info
        elif gene_id in AS_genes_test_list:
            AS_test_actual_junction[gene_id] = info
        elif gene_id in no_AS_genes_train_list:
            no_AS_train_actual_junction[gene_id] = info
        elif gene_id in no_AS_genes_test_list:
            no_AS_test_actual_junction[gene_id] = info
        else:
            if gene_id in mrna_gene_set_final:
                other_gene_actual_junction[gene_id] = info

with open(f'{working_dir}/{PLANT_NAME}_AS_train_actualjunction.txt', 'w') as outfile:
    for gene_id, info in AS_train_actual_junction.items():
        outfile.write(f'{gene_id}\t{info}\n')

with open(f'{working_dir}/{PLANT_NAME}_AS_test_actualjunction.txt', 'w') as outfile:
    for gene_id, info in AS_test_actual_junction.items():
        outfile.write(f'{gene_id}\t{info}\n')

with open(f'{working_dir}/{PLANT_NAME}_no_AS_train_actualjunction.txt', 'w') as outfile:
    for gene_id, info in no_AS_train_actual_junction.items():
        outfile.write(f'{gene_id}\t{info}\n')

with open(f'{working_dir}/{PLANT_NAME}_no_AS_test_actualjunction.txt', 'w') as outfile:
    for gene_id, info in no_AS_test_actual_junction.items():
        outfile.write(f'{gene_id}\t{info}\n')

with open(f'{working_dir}/{PLANT_NAME}_other_gene_actualjunction.txt', 'w') as outfile:
    for gene_id, info in other_gene_actual_junction.items():
        outfile.write(f'{gene_id}\t{info}\n')

In [ ]:
# ----------------------------
# Step 10: Generate Testing Files for Top-k Accuracy Measurement
# ----------------------------
no_AS_test_gene_sequences = []
AS_test_gene_sequences = []

# Load genome sequences into a dictionary (key: chromosome ID)
genome_sequences = {}
for seq_record in SeqIO.parse(f'{working_dir}/{PLANT_NAME}.fna', "fasta"):
    genome_sequences[seq_record.id] = seq_record.seq.upper()

# Process no-AS test genes and compute intron indices relative to gene start
with open(f'{working_dir}/{PLANT_NAME}_no_AS_test_actualjunction.txt', 'r') as infile:
    for line in tqdm(infile, desc="Loading no-AS test genes", ascii=True):
        parts = line.strip().split("\t")
        gene_id = parts[0]
        info = ast.literal_eval(parts[1])
        chromosome = info[0]
        strand = info[1]
        gene_range = info[2]
        intron_start_list = info[4]
        intron_end_list = info[5]
        filtered_sites = info[7]
        gene_start = int(gene_range[0])
        gene_end = int(gene_range[1])
        new_intron_start_indices = []
        new_intron_end_indices = []

        if strand == "+":
            for site in intron_start_list:
                if site in filtered_sites:
                    new_intron_start_indices.append(site - gene_start)
            for site in intron_end_list:
                if site in filtered_sites:
                    new_intron_end_indices.append(site - gene_start + 1)
            seq = genome_sequences[chromosome][gene_start - 1:gene_end]
        else:
            for site in intron_start_list:
                if site in filtered_sites:
                    new_intron_start_indices.append(gene_end - site)
            for site in intron_end_list:
                if site in filtered_sites:
                    new_intron_end_indices.append(gene_end - site + 1)
            seq = genome_sequences[chromosome][gene_start -
                                               1:gene_end].reverse_complement()

        new_intron_start_indices = sorted(new_intron_start_indices)
        new_intron_end_indices = sorted(new_intron_end_indices)
        start_indices_str = ",".join(str(idx)
                                     for idx in new_intron_start_indices)
        end_indices_str = ",".join(str(idx) for idx in new_intron_end_indices)
        seq_id = f'{gene_id}_{chromosome}_{strand}_{gene_start}_{gene_end}_{start_indices_str}_{end_indices_str}'
        new_record = SeqRecord(seq, id=seq_id, description="")
        no_AS_test_gene_sequences.append(new_record)

with open(f'{working_dir}/{PLANT_NAME}_no_AS_test_gene_sequences.fasta', "w") as output_handle:
    for record in tqdm(no_AS_test_gene_sequences, desc="Writing no-AS test sequences"):
        output_handle.write(f">{record.id}\n{record.seq}\n")

# Process AS test genes similarly
with open(f'{working_dir}/{PLANT_NAME}_AS_test_actualjunction.txt', 'r') as infile:
    for line in tqdm(infile, desc="Loading AS test genes", ascii=True):
        parts = line.strip().split("\t")
        gene_id = parts[0]
        info = ast.literal_eval(parts[1])
        chromosome = info[0]
        strand = info[1]
        gene_range = info[2]
        intron_start_list = info[4]
        intron_end_list = info[5]
        filtered_sites = info[7]
        gene_start = int(gene_range[0])
        gene_end = int(gene_range[1])
        new_intron_start_indices = []
        new_intron_end_indices = []

        if strand == "+":
            for site in intron_start_list:
                if site in filtered_sites:
                    new_intron_start_indices.append(site - gene_start)
            for site in intron_end_list:
                if site in filtered_sites:
                    new_intron_end_indices.append(site - gene_start + 1)
            seq = genome_sequences[chromosome][gene_start - 1:gene_end]
        else:
            for site in intron_start_list:
                if site in filtered_sites:
                    new_intron_start_indices.append(gene_end - site)
            for site in intron_end_list:
                if site in filtered_sites:
                    new_intron_end_indices.append(gene_end - site + 1)
            seq = genome_sequences[chromosome][gene_start -
                                               1:gene_end].reverse_complement()

        new_intron_start_indices = sorted(new_intron_start_indices)
        new_intron_end_indices = sorted(new_intron_end_indices)
        start_indices_str = ",".join(str(idx)
                                     for idx in new_intron_start_indices)
        end_indices_str = ",".join(str(idx) for idx in new_intron_end_indices)
        seq_id = f'{gene_id}_{chromosome}_{strand}_{gene_start}_{gene_end}_{start_indices_str}_{end_indices_str}'
        new_record = SeqRecord(seq, id=seq_id, description="")
        AS_test_gene_sequences.append(new_record)

with open(f'{working_dir}/{PLANT_NAME}_AS_test_gene_sequences.fasta', "w") as output_handle:
    for record in tqdm(AS_test_gene_sequences, desc="Writing AS test sequences"):
        output_handle.write(f">{record.id}\n{record.seq}\n")

In [ ]:
# ----------------------------
# Step 11: Generate Training Files for UniSplicer Model Training
# ----------------------------

# ------------------------------------------------------------------
# 1. One-hot utilities
# ------------------------------------------------------------------
def average_tensors(*tensors):
    """Return the element-wise mean of an arbitrary number of tensors."""
    return sum(tensors) / len(tensors)


# Basic one-hot encodings for the four canonical bases (+ unknown “N”)
NUCLEOTIDE_ONE_HOT = {
    "A": torch.tensor([1., 0., 0., 0.]),
    "T": torch.tensor([0., 1., 0., 0.]),
    "C": torch.tensor([0., 0., 1., 0.]),
    "G": torch.tensor([0., 0., 0., 1.]),
    "N": torch.tensor([0., 0., 0., 0.])     # padding / unknown
}

# Ambiguous IUPAC codes represented by the average of relevant bases
AMBIGUOUS_BASES = {
    # A or G
    "R": average_tensors(NUCLEOTIDE_ONE_HOT["A"], NUCLEOTIDE_ONE_HOT["G"]),
    # C or T
    "Y": average_tensors(NUCLEOTIDE_ONE_HOT["C"], NUCLEOTIDE_ONE_HOT["T"]),
    # G or T
    "K": average_tensors(NUCLEOTIDE_ONE_HOT["G"], NUCLEOTIDE_ONE_HOT["T"]),
    # G or C
    "S": average_tensors(NUCLEOTIDE_ONE_HOT["G"], NUCLEOTIDE_ONE_HOT["C"]),
    # A or T
    "W": average_tensors(NUCLEOTIDE_ONE_HOT["A"], NUCLEOTIDE_ONE_HOT["T"]),
    # A or C
    "M": average_tensors(NUCLEOTIDE_ONE_HOT["A"], NUCLEOTIDE_ONE_HOT["C"]),
    "B": average_tensors(NUCLEOTIDE_ONE_HOT["C"], NUCLEOTIDE_ONE_HOT["G"], NUCLEOTIDE_ONE_HOT["T"]),
    "D": average_tensors(NUCLEOTIDE_ONE_HOT["A"], NUCLEOTIDE_ONE_HOT["G"], NUCLEOTIDE_ONE_HOT["T"]),
    "H": average_tensors(NUCLEOTIDE_ONE_HOT["A"], NUCLEOTIDE_ONE_HOT["C"], NUCLEOTIDE_ONE_HOT["T"]),
    "V": average_tensors(NUCLEOTIDE_ONE_HOT["A"], NUCLEOTIDE_ONE_HOT["C"], NUCLEOTIDE_ONE_HOT["G"]),
}
NUCLEOTIDE_ONE_HOT.update(AMBIGUOUS_BASES)   # merge dictionaries in-place

# One-hot encoding for splice labels:
#   0 = no-splice, 1 = acceptor, 2 = donor, 3 = padding/unknown
SPLICE_LABEL_ONE_HOT = torch.tensor([
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 1],
    [0, 0, 0]
], dtype=torch.float32)


# ------------------------------------------------------------------
# 2. Encoding helpers
# ------------------------------------------------------------------
def encode_sequence_windows(seq_windows):
    """
    Convert an array of sequence windows (str) → tensor [N, L, 4].
    Unknown characters fall back to the ‘N’ vector.
    """
    encoded = []
    for window in seq_windows:
        encoded_window = [NUCLEOTIDE_ONE_HOT.get(b, NUCLEOTIDE_ONE_HOT["N"])
                          for b in window]
        encoded.append(torch.stack(encoded_window))
    return torch.stack(encoded)


def encode_label_windows(label_windows):
    """
    Convert raw integer label windows → tensor [N, flank_len, 3].
    Values outside {0,1,2} are treated as padding (index 3).
    """
    encoded_rows = []
    for row in label_windows[0]:
        encoded_row = [SPLICE_LABEL_ONE_HOT[int(lbl) if lbl in (0, 1, 2) else 3]
                       for lbl in row]
        encoded_rows.append(torch.stack(encoded_row))
    return torch.stack(encoded_rows)


# ------------------------------------------------------------------
# 3. Window creator
# ------------------------------------------------------------------
def create_windows_and_labels(seq, gene_start, gene_end,
                              intron_starts, intron_ends, window_context):
    """
    Split a gene into sliding windows with context and produce
    one-hot encoded sequence & label tensors.
    """
    flank_len = window_context // 2
    length_gene = gene_end - gene_start + 1

    # Pad with ‘N’ so windows at the ends still have full context
    padded_seq = 'N' * flank_len + seq + 'N' * flank_len

    # Parse comma-separated intron position strings (may be empty)
    intron_starts = [] if intron_starts.strip() == '' else [
        int(i) for i in intron_starts.split(',')
        if i.strip() and int(i) < length_gene
    ]
    intron_ends = [] if intron_ends.strip() == '' else [
        int(i) for i in intron_ends.split(',')
        if i.strip() and int(i) < length_gene
    ]

    # Base-level arrays
    seq_array = list(padded_seq)
    label_array = [-0] * length_gene          # default → padding

    # Mark donors / acceptors (0-based within gene)
    for pos in intron_starts:
        label_array[pos] = 2                  # donor
    for pos in intron_ends:
        label_array[pos] = 1                  # acceptor

    # Number of sliding windows (stride = flank_len)
    num_windows = ceil(len(label_array) / flank_len)

    # Pad arrays so final window is complete
    label_array = np.pad(label_array, (0, flank_len), constant_values=-1)
    seq_array = np.pad(np.array(seq_array, dtype=object),
                       (0, flank_len), constant_values='N')

    # Allocate windows
    seq_windows = np.full(
        (num_windows, flank_len + window_context), 'N', dtype=object)
    label_windows = -np.ones((num_windows, flank_len))

    # Fill windows
    for w in range(num_windows):
        start = flank_len * w
        seq_windows[w] = seq_array[start: start + flank_len + window_context]
        label_windows[w] = label_array[start: start + flank_len]

    # Encode & return tensors
    encoded_seq = encode_sequence_windows(seq_windows)
    encoded_labels = encode_label_windows([label_windows])
    return encoded_seq, encoded_labels


# ------------------------------------------------------------------
# 4. Derive intron coordinates from exon GFF
# ------------------------------------------------------------------
exons = defaultdict(list)
intron_length = []

with open(f'{working_dir}/{PLANT_NAME}/{PLANT_NAME}_exon.gff', "r") as file:
    for line in file:
        column = line.strip().split('\t')
        chromosome = column[0]
        region = column[1]
        start = int(column[2])
        end = int(column[3])
        strand = column[4]
        annotation = column[5]

        # Extract transcript ID
        mrna = annotation.split("parent=rna-")[1].split(";")[0]

        # Store exon coords keyed by transcript
        if "gbkey=mrna" in line and region == "exon":
            exons[mrna].append((chromosome, start, end, strand, annotation))

# Write introns to new GFF and collect lengths
with open(f'{working_dir}/{PLANT_NAME}/{PLANT_NAME}_intron.gff', "w") as outfile:
    for gene_id, gene_exons in exons.items():
        # Sort exons in transcription order (strand-aware)
        gene_exons.sort(key=lambda x: x[1] if x[3] == '+' else -x[1])

        # Consecutive exon pairs → one intron
        for i in range(len(gene_exons) - 1):
            exon1, exon2 = gene_exons[i], gene_exons[i + 1]

            if gene_exons[1][3] == "+":   # forward strand
                intron_start_, intron_end_ = exon1[2] + 1, exon2[1] - 1
            else:                         # reverse strand
                intron_start_, intron_end_ = exon2[2] + 1, exon1[1] - 1

            intron_info = [
                exon1[0], "intron", str(intron_start_), str(intron_end_),
                exon1[3], exon1[-1]
            ]
            length_intron = intron_end_ - intron_start_ + 1
            intron_length.append(length_intron)

            outfile.write("\t".join(intron_info) + "\n")


# ------------------------------------------------------------------
# 5. Choose window context size based on intron median length
# ------------------------------------------------------------------
intron_median = statistics.median(intron_length)

if intron_median < 500:
    window_context = 600
elif 500 <= intron_median < 1000:
    window_context = 1200
elif 1000 <= intron_median < 1500:
    window_context = 1800
else:                                   # ≥ 1500 bp
    window_context = 2400

print(f'{species} intron median length is: {intron_median}, '
      f'choose window context length to be: {window_context}')


# ------------------------------------------------------------------
# 6. Merge individual junction files → combined training file
# ------------------------------------------------------------------
input_files = [
    (f"{working_dir}/{PLANT_NAME}/{PLANT_NAME}_no_AS_train_actualjunction.txt", "CS"),
    (f"{working_dir}/{PLANT_NAME}/{PLANT_NAME}_AS_train_actualjunction.txt",    "AS"),
    (f"{working_dir}/{PLANT_NAME}/{PLANT_NAME}_other_gene_actualjunction.txt",  "OR")
]
training_junction_file = f"{working_dir}/{PLANT_NAME}/{PLANT_NAME}_train_combined_actualjunction.txt"

with open(training_junction_file, "w") as outfile:
    for file_path, label in input_files:
        with open(file_path, "r") as infile:
            for line in infile:
                line = line.strip()
                if not line:
                    continue                      # skip empty lines
                parts = line.split("\t", 1)
                if len(parts) == 2:
                    gene_id, info = parts
                    outfile.write(f"{gene_id}\t{label}\t{info}\n")


# ------------------------------------------------------------------
# 7. Load genome FASTA → dict {chromosome: Seq}
# ------------------------------------------------------------------
Gene_name, Chromosome, Strand = [], [], []
Gene_start, Gene_end = [], []
Intron_start, Intron_end = [], []
Sequence, Source = [], []
genome_sequences = {}

for seq_record in SeqIO.parse(f'{working_dir}/{PLANT_NAME}/{PLANT_NAME}.fna', "fasta"):
    genome_sequences[seq_record.id] = seq_record.seq.upper()


# ------------------------------------------------------------------
# 8. Parse combined junction file & build per-gene records
# ------------------------------------------------------------------
with open(training_junction_file, 'r') as filein:
    for line in tqdm(filein, desc="Loading", ascii=True):
        feature = line.strip().split("\t")
        gene_id = feature[0]
        source = feature[1]
        info = ast.literal_eval(feature[2])

        chromosome = info[0]
        strand = info[1]
        gene_range = info[2]
        intron_start = info[4]
        intron_end = info[5]
        actual_site = info[6]     # all annotated splice sites
        filtered_actual_site = info[7]  # quality-filtered subset

        gene_start = int(gene_range[0])
        gene_end = int(gene_range[1])

        new_index_intron_start, new_index_intron_end = [], []

        # Skip genes with no introns
        if intron_start and intron_end:
            # --------------------------------------------------
            # 8-A. Compute intron indices relative to gene start
            # --------------------------------------------------
            if strand == "+":               # forward strand
                for item in intron_start:
                    if item in filtered_actual_site:
                        new_index_intron_start.append(int(item) - gene_start)
                for item in intron_end:
                    if item in filtered_actual_site:
                        new_index_intron_end.append(int(item) - gene_start + 1)

                seq = genome_sequences[chromosome][gene_start - 1: gene_end]

            else:                           # reverse strand
                for item in intron_start:
                    if item in filtered_actual_site:
                        new_index_intron_start.append(gene_end - item)
                for item in intron_end:
                    if item in filtered_actual_site:
                        new_index_intron_end.append(gene_end - item + 1)

                seq = genome_sequences[chromosome][gene_start - 1: gene_end] \
                    .reverse_complement()

            # Sort & stringify index lists
            new_index_intron_start = sorted(new_index_intron_start)
            new_index_intron_end = sorted(new_index_intron_end)

            index_intron_start_string = ",".join(
                str(item) for item in new_index_intron_start)
            index_intron_end_string = ",".join(
                str(item) for item in new_index_intron_end)

            # Build composite FASTA header
            id_seq = (f'{gene_id}_{chromosome}_{strand}_{gene_start}_{gene_end}_'
                      f'{index_intron_start_string}_{index_intron_end_string}')
            new_record = SeqIO.SeqRecord(seq, id=id_seq, description="")

            # Append to master lists
            Gene_name.append(gene_id)
            Chromosome.append(chromosome)
            Strand.append(strand)
            Gene_start.append(gene_start)
            Gene_end.append(gene_end)
            Intron_start.append(index_intron_start_string)
            Intron_end.append(index_intron_end_string)
            Sequence.append(str(seq))
            Source.append(source)


# ------------------------------------------------------------------
# 9. Encode sequences & labels → write HDF5 dataset
# ------------------------------------------------------------------
total_genes = len(Gene_name)

h5_path = (f'{working_dir}/{PLANT_NAME}/'
           f'{PLANT_NAME}_training_dataset_window_context_{window_context}.h5')

with h5py.File(h5_path, "w") as h5_out:
    for gene_idx in tqdm(range(total_genes), desc="Encoding genes"):
        seq = Sequence[gene_idx]
        strand = Strand[gene_idx]          # (kept for completeness)
        intron_starts = Intron_start[gene_idx]
        intron_ends = Intron_end[gene_idx]
        gene_start_idx = int(Gene_start[gene_idx])
        gene_end_idx = int(Gene_end[gene_idx])

        # Create windowed tensors
        seq_tensor, label_tensor = create_windows_and_labels(
            seq, gene_start_idx, gene_end_idx,
            intron_starts, intron_ends, window_context
        )
        label_tensor = label_tensor.unsqueeze(0)   # add sample dim: [1, …]

        # Write tensors: I* = input, L* = labels
        h5_out.create_dataset(f"I{gene_idx}",
                              data=seq_tensor.numpy().astype("int8"))
        h5_out.create_dataset(f"L{gene_idx}",
                              data=label_tensor.numpy().astype("int8"))